# La consulta no puede ver la pregunta

### El alcance de una convolución corta decide qué parte de una pregunta condiciona la búsqueda

**Maximiliano Speranza** · Investigador independiente, Buenos Aires
[ORCID 0009-0005-0413-8554](https://orcid.org/0009-0005-0413-8554)

---

Este notebook demuestra, **sobre un modelo público y sin entrenar nada**, que en los modelos de
atención lineal y de espacio de estados hay una parte de cada pregunta que la búsqueda interna
**literalmente no puede ver**, y que lo que queda afuera no se degrada de a poco: **desaparece
exacto**.

Todo lo que sigue corre en CPU en un par de minutos.

| | |
|---|---|
| **Qué es** | un límite duro sobre qué parte de la pregunta condiciona la recuperación |
| **Cómo funciona** | la consulta se arma con una convolución causal corta, y su alcance es la ventana |
| **Para qué sirve** | da un diagnóstico que no necesita entrenamiento, y un arreglo que cuesta el 0,12 % de los parámetros |
| **Cómo se descubrió** | cambiando **un** token de la pregunta y midiendo si la búsqueda se mueve |


## 0 · Preparación

Sólo hace falta `transformers` y `torch`. El modelo son 130M de parámetros y corre en CPU.


In [ ]:
!pip -q install transformers torch --upgrade


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

NOMBRE = 'state-spaces/mamba-130m-hf'
tok = AutoTokenizer.from_pretrained(NOMBRE)
modelo = AutoModelForCausalLM.from_pretrained(NOMBRE).eval()
capas = modelo.backbone.layers

print(f'{NOMBRE} · {sum(p.numel() for p in modelo.parameters()):,} parámetros · {len(capas)} capas')


## 1 · Dónde vive la ventana

En esta familia de modelos el estado recurrente **sí** ve la secuencia entera. Pero la *consulta*
con la que se lee ese estado no se arma con la secuencia entera: se arma con una **convolución
causal corta** aplicada antes de calcular las proyecciones de lectura.

El tamaño de núcleo por omisión es **4**. Es decir que la consulta de una capa es función del token
actual y de **tres anteriores**, y de nada más.

Ese número no suele medirse: se hereda. Mamba-2 usa 4 por defecto, y la implementación de referencia
de Qwen3-Next expone `linear_conv_kernel_dim` documentado como «kernel size of the convolution used
in linear attention layers», también con valor 4.

Miremos el peso real.


In [ ]:
w = capas[0].mixer.conv1d.weight    # (canales, 1, kernel)
print('forma del peso de la convolución:', tuple(w.shape))
print('canales:', w.shape[0], '· tamaño de núcleo:', w.shape[2])


## 2 · Primera medición · el alcance real no es el nominal

El núcleo dice 4, o sea alcance 3 hacia atrás. Pero un peso puede estar en cero, y entonces el
alcance efectivo es menor. Vale la pena **medirlo en vez de suponerlo**.


In [ ]:
print('capa | max|peso| de cada tap, del más viejo al actual')
for i in (0, 1, 12, 23):
    taps = capas[i].mixer.conv1d.weight[:, 0, :].abs().max(dim=0).values
    print(f'  {i:>3} | ' + '  '.join(f'{t.item():.6f}' for t in taps))

ceros = sum(1 for c in capas if c.mixer.conv1d.weight[:, 0, 0].abs().max().item() == 0.0)
print(f'\ncapas donde el tap MÁS VIEJO es exactamente cero: {ceros} de {len(capas)}')


**El tap más viejo es cero exacto en las 24 capas.** No pequeño: cero, en las 1.536 entradas de
cada capa.

Así que el alcance efectivo de este checkpoint es **2 tokens hacia atrás**, no 3. Reportamos la
medición y no una causa: un peso exactamente nulo en 24 × 1536 entradas no sale de un gradiente, así
que un artefacto de conversión es plausible, pero no lo verificamos y no lo afirmamos.

La consecuencia práctica, si esto generaliza, es que **el núcleo nominal es una cota superior del
alcance y hay que medirlo**.


## 3 · Segunda medición · el corte es exacto, no gradual

Ahora la intervención que define todo el trabajo, y que es tan simple que se puede repetir en
cualquier arquitectura:

> **Cambiá un token de la pregunta y mirá si la búsqueda se mueve.**

Tomamos una posición de lectura, cambiamos **un solo token** a distancia $d$ hacia atrás, y medimos
cuánto se movió la salida de la convolución en esa posición.


In [ ]:
texto = 'The capital of the country that borders Spain and lies on the Atlantic is'
ids = tok(texto, return_tensors='pt').input_ids
n = ids.shape[1]
pos_lectura = n - 1

capt = {}
gancho = capas[0].mixer.conv1d.register_forward_hook(
    lambda mod, ent, sal: capt.__setitem__('conv', sal.detach().clone()))

def conv_en(x):
    capt.clear()
    with torch.no_grad():
        modelo(x)
    return capt['conv']

base = conv_en(ids)
OTRO, ALT = tok(' dog').input_ids[0], tok(' cat').input_ids[0]

print('distancia | movimiento de la CONSULTA al cambiar un token')
for d in range(1, 8):
    p = pos_lectura - d
    if p < 1: break
    alt = ids.clone()
    alt[0, p] = OTRO if alt[0, p] != OTRO else ALT
    mov = (conv_en(alt)[0, :, pos_lectura] - base[0, :, pos_lectura]).abs().max().item()
    marca = '  <-- SE MUEVE' if mov > 0 else '  <-- CERO EXACTO, es invisible'
    print(f'   d = {d}    {mov:.6f}{marca}')


Eso es el resultado central. **No hay decaimiento**: el movimiento pasa de un valor grande a
`0.000000` de un paso al siguiente, exactamente donde termina el alcance medido.

Un token que cae afuera de esa ventana no influye poco en la búsqueda. **No influye nada.**


## 4 · La disociación · el modelo ve el contexto, la consulta no

Acá está la parte que sorprende, y es la que hace que esto no sea obvio. Si la convolución no ve ese
token, ¿el modelo lo ignora?

No. El **estado** sí lo ve, porque la recurrencia lo arrastra. Lo comprobamos midiendo, con la misma
intervención, cuánto se mueve la **salida de la capa** en vez de la de la convolución.


In [ ]:
capt2 = {}
g2 = capas[0].register_forward_hook(
    lambda mod, ent, sal: capt2.__setitem__('capa',
        (sal[0] if isinstance(sal, tuple) else sal).detach().clone()))

def ambos(x):
    capt.clear(); capt2.clear()
    with torch.no_grad():
        modelo(x)
    return capt['conv'], capt2['capa']

bc, bl = ambos(ids)
print('distancia |   consulta   |  salida de la capa')
for d in range(1, 8):
    p = pos_lectura - d
    if p < 1: break
    alt = ids.clone()
    alt[0, p] = OTRO if alt[0, p] != OTRO else ALT
    c, l = ambos(alt)
    mc = (c[0, :, pos_lectura] - bc[0, :, pos_lectura]).abs().max().item()
    ml = (l[0, pos_lectura, :] - bl[0, pos_lectura, :]).abs().max().item()
    print(f'   d = {d}    {mc:>10.6f}    {ml:>10.6f}')

gancho.remove(); g2.remove()


**La salida de la capa se mueve a TODAS las distancias. La consulta, no.**

> El estado ve la secuencia entera; la consulta que lo lee, no.

Por eso el problema es invisible desde afuera: el modelo *tiene* la información. Lo que no puede es
**usarla para decidir qué buscar**.


## 5 · Por qué importa · la falla concreta que produce

Pensemos una pregunta de la forma:

> *«cuál es la **⟨relación⟩** de **⟨entidad⟩**?»*

Leída desde el final, la **entidad** queda a 1 token y la **relación** a 3. Con alcance 2, la
relación cae **exactamente un token afuera**, de forma determinista, en el 100 % de las consultas.

El modelo entonces busca en su memoria **usando sólo la entidad**. Y ahí aparece la falla que más se
parece a una alucinación real: si le preguntás por una relación que **nunca se enunció**, sobre una
entidad que **sí** existe, no se abstiene. Recupera el hecho vecino, el que sí tiene guardado de esa
entidad, y lo entrega con confianza.

No estaba ignorando la relación. **No la podía ver.**

### ¿Y esto pasa en preguntas reales?

Sí, y es la parte más aplicable. Medido sobre **33.585 preguntas** de cuatro corpus públicos (SQuAD,
Natural Questions, TriviaQA y HotpotQA), entre el **90 % y el 100 %** de las preguntas reales tienen
sus partes discriminantes más separadas que el alcance medido. Y **empeora con la dificultad**: en
las preguntas multi-hop llega a **1,0000**.

La configuración no es un caso de borde armado en un laboratorio. Es el caso ordinario.


## 6 · La receta · un diagnóstico que no necesita entrenamiento

Todo lo anterior se resume en una función que se puede aplicar a cualquier modelo que consulte una
memoria a través de una consulta formada localmente.


In [ ]:
def alcance_de_la_consulta(modelo, tok, texto, capa=0, dmax=8):
    """Devuelve hasta qué distancia un token todavía influye en la consulta."""
    ids = tok(texto, return_tensors='pt').input_ids
    pos = ids.shape[1] - 1
    caja = {}
    h = modelo.backbone.layers[capa].mixer.conv1d.register_forward_hook(
        lambda m, e, s: caja.__setitem__('c', s.detach().clone()))
    def q(x):
        caja.clear()
        with torch.no_grad(): modelo(x)
        return caja['c'][0, :, pos]
    base, alcance = q(ids), 0
    a, b = tok(' dog').input_ids[0], tok(' cat').input_ids[0]
    for d in range(1, dmax + 1):
        p = pos - d
        if p < 1: break
        alt = ids.clone(); alt[0, p] = a if alt[0, p] != a else b
        if (q(alt) - base).abs().max().item() > 0: alcance = d
    h.remove()
    return alcance

print('alcance medido de la consulta:', alcance_de_la_consulta(modelo, tok, texto), 'tokens')


**Cómo se lee el resultado.** Si cambiar un token no mueve la búsqueda, ese token es **invisible**
para la búsqueda, y cualquier confianza que el modelo exprese sobre él **no está ganada**.

No hace falta entrenar, ni calcular gradientes, ni tener etiquetas.


## 7 · El arreglo, y lo que cuesta

El arreglo es tan directo como el diagnóstico: **ensanchar el núcleo de la convolución que forma la
consulta**, de 3 a 5, con lo que el alcance pasa de 2 a 4 y la parte que faltaba entra en la ventana.

Medido en un modelo chico entrenado desde cero para poder controlar la distancia:

| | núcleo 3 | núcleo 5 |
|---|---|---|
| abstención correcta en el caso difícil | 0,5850 – 0,7349 | **0,9931 – 1,0000** |
| exactitud global | 0,91 – 0,93 | **0,988 – 0,993** |

Tres semillas cada uno, **sin un solo solape** entre condiciones, contra un piso trivial de 0,4065.

**Y cuesta 1.024 parámetros sobre 865.651, o sea el 0,12 % del modelo.** Dos taps más, por 128
canales, por 4 bloques.

No es capacidad. Es acceso.


## Lo que esto NO afirma

Vale la pena ser explícito, porque el límite es parte del resultado.

1. **En un modelo profundo la ventana atenúa, no bloquea.** El corte es exacto en la capa 0, pero
   desde la capa 1 la recurrencia restituye la señal atenuada. Medimos la tasa: 1,077 y 1,028 por
   token en modelos de 24 y 48 capas, con $r = 0,978$ en los dos.
2. **El efecto sobre el comportamiento es transitorio cuando hay capas de sobra.** Al ajustar un
   modelo de 24 capas, la diferencia existe en el paso 100 y **desaparece del todo para el paso 400**.
   Es un negativo, estaba pre-registrado, y lo informamos.

> **Donde no quedan capas con que pagar, la ventana pone un techo. Donde quedan, pone un peaje.**

Así que la afirmación fuerte vale para **memoria consultada desde una capa temprana**, y no como
predicción de comportamiento para un modelo profundo entrenado hasta converger.

---

### En una línea

**La consulta con la que se busca en una memoria tiene que formarse donde ya se vio la pregunta
completa.** Y si no estás seguro de que así sea, cambiá un token y fijate si la búsqueda se mueve.
